# 01 — EDA & Tien Xu Ly Du Lieu
Pipeline: Load -> Kiem tra chat luong -> Lam sach -> EDA

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.data.loader  import load_config, load_raw
from src.data.cleaner import DataCleaner
from src.visualization.plots import (
    plot_quality_check, plot_before_after,
    plot_yield_distribution, plot_correlation
)

cfg = load_config("../configs/params.yaml")
NUM_COLS = ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Yield"]
print("Config loaded OK")

## 1.1 Load du lieu tho

In [ ]:
df_raw = load_raw(cfg)
print(f"Shape: {df_raw.shape}")
df_raw.head(10)

## 1.2 Data Dictionary

In [ ]:
data_dict = {
    "Area":    "Quoc gia / vung dia ly",
    "Item":    "Loai cay trong (10 loai)",
    "Year":    "Nam thu thap (1990-2013)",
    "average_rain_fall_mm_per_year": "Luong mua TB (mm/nam)",
    "pesticides_tonnes": "Luong thuoc tru sau (tan)",
    "avg_temp": "Nhiet do TB (do C)",
    "Yield":   "Nang suat (hg/ha) --- TARGET",
}
pd.DataFrame(list(data_dict.items()), columns=["Cot", "Y nghia"])

## 1.3 Kiem tra chat luong du lieu

In [ ]:
print("Kieu du lieu:")
print(df_raw.dtypes)
print("\nThong ke mo ta:")
print(df_raw.describe().round(2).to_string())

In [ ]:
null_df = pd.DataFrame({
    "So_NULL":   df_raw.isnull().sum(),
    "Phan_tram": (df_raw.isnull().sum()/len(df_raw)*100).round(2)
})
print("Gia tri NULL:")
print(null_df[null_df["So_NULL"]>0])
print(f"\nBan ghi trung lap: {df_raw.duplicated().sum()}")
for col in NUM_COLS:
    n_neg = (df_raw[col] < 0).sum()
    print(f"  Gia tri am {col}: {n_neg}")

In [ ]:
plot_quality_check(df_raw, NUM_COLS, cfg["paths"]["figures"])
plt.show()

## 1.4 Lam sach du lieu

In [ ]:
cleaner  = DataCleaner(cfg)
df_clean = cleaner.fit_transform(df_raw)
print("\nBao cao lam sach:")
print(cleaner.get_report().to_string(index=False))

In [ ]:
plot_before_after(df_raw, df_clean, NUM_COLS, cfg["paths"]["figures"])
plt.show()

## 1.5 EDA

In [ ]:
plot_yield_distribution(df_clean, cfg["paths"]["figures"])
plt.show()
print(f"Skewness goc:  {df_clean['Yield'].skew():.3f}")
print(f"Skewness log:  {np.log1p(df_clean['Yield']).skew():.3f}")

In [ ]:
plot_correlation(df_clean, NUM_COLS, cfg["paths"]["figures"])
plt.show()

In [ ]:
crop_stats = df_clean.groupby("Item")["Yield"].agg(["mean","std"]).sort_values("mean", ascending=False)
fig, ax = plt.subplots(figsize=(12,4))
ax.bar(crop_stats.index, crop_stats["mean"],
       color=sns.color_palette("Set2", len(crop_stats)), edgecolor="white")
ax.errorbar(range(len(crop_stats)), crop_stats["mean"], yerr=crop_stats["std"],
            fmt="none", color="gray", capsize=4)
ax.set_xticks(range(len(crop_stats)))
ax.set_xticklabels(crop_stats.index, rotation=35, ha="right")
ax.set_title("Nang suat TB theo loai cay (+/- std)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
import pathlib
pathlib.Path("../data/processed").mkdir(parents=True, exist_ok=True)
df_clean.to_parquet("../data/processed/yield_clean.parquet", index=False)
print(f"Saved: yield_clean.parquet  ({len(df_clean):,} rows)")